# Load exported results (`.nc`, `.h5`, `.npz`)

**Export Results** (Sequence mode) writes the result of the *current*
simulation as plain data files: xarray NetCDF (`.nc`, default, with an
analysis notebook), HDF5 (`.h5`) or NumPy (`.npz`), and optionally
Bruker / DICOM / NIfTI images. These files

* can be read **without Bloch Simulator** (xarray, h5py, NumPy, MATLAB),
* contain the signal, k-space, reconstructed images, final magnetization,
  B1 maps and the simulation parameters as JSON metadata,
* **cannot** be re-opened as a GUI session. They do not contain the phantom
  object, the B1 field files or the GUI controls. Use a Bloch project
  ([load_bloch_project.ipynb](load_bloch_project.ipynb)) for that.

Free Mode exports (HDF5 with `mx`, `my`, `mz` per time, position and
frequency) are covered by [read_hdf5_examples.py](read_hdf5_examples.py).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# Change this to your exported file (.nc, .h5 or .npz).
RESULT_FILE = Path("result.nc")

## NetCDF (`.nc`)

NetCDF has no complex data type, so complex variables (signal, k-space,
images) are stored as `<name>_real` and `<name>_imag`. The helper joins them
again. The dimensions are named, e.g. `adc`, `coil`, `pool`, `phase_y`,
`read_x`.

In [ ]:
def load_netcdf(filename):
    """Return the xarray Dataset (complex variables restored) and its metadata."""
    with xr.open_dataset(filename) as stored:
        dataset = stored.load()
    for name in list(dataset.data_vars):
        base = name[:-5]
        if name.endswith("_real") and f"{base}_imag" in dataset:
            dataset[base] = dataset[name] + 1j * dataset[f"{base}_imag"]
            dataset = dataset.drop_vars([name, f"{base}_imag"])
    metadata = json.loads(dataset.attrs.get("metadata_json", "{}"))
    return dataset, metadata


if RESULT_FILE.suffix == ".nc":
    dataset, metadata = load_netcdf(RESULT_FILE)
    print(sorted(dataset.data_vars))

## HDF5 (`.h5`) and NumPy (`.npz`)

Both hold the same arrays as plain datasets and the metadata as one JSON
string `metadata_json` (a file attribute in HDF5, an entry in NPZ).

In [ ]:
def load_arrays(filename):
    """Return a dict of arrays and the metadata from an HDF5 or NPZ export."""
    if Path(filename).suffix == ".npz":
        with np.load(filename, allow_pickle=False) as stored:
            arrays = {key: stored[key] for key in stored.files}
        metadata = json.loads(str(arrays.pop("metadata_json")))
    else:
        import h5py

        with h5py.File(filename, "r") as handle:
            arrays = {key: handle[key][()] for key in handle.keys()}
            metadata = json.loads(handle.attrs["metadata_json"])
    return arrays, metadata


if RESULT_FILE.suffix in (".h5", ".npz"):
    arrays, metadata = load_arrays(RESULT_FILE)
    for name, values in sorted(arrays.items()):
        print(f"{name}: {np.shape(values)} {np.asarray(values).dtype}")

## Simulation parameters

The metadata is the same for all three formats.
`metadata["sequence_definitions"]` holds the sequence parameters (flip angle
in degrees, matrix, FOV in m, TR/TE in s, …). Arrays are stored in their text
form, e.g. `"[64. 64.]"`. Nucleus, B0, the number of simulated spins and, for
a run of a parameter loop, `parameter_loop` are stored next to it.

In [ ]:
definitions = metadata.get("sequence_definitions", {})
print("Sequence:", definitions.get("Name", metadata.get("sequence_source")))
for key in ("FlipAngleDeg", "MatrixSize", "FOV", "TR", "TE"):
    print(f"  {key}: {definitions.get(key)}")
print("Nucleus:", metadata.get("nucleus"), " B0:", metadata.get("field_strength_t"), "T")
print("Simulated spins:", metadata.get("n_simulated_spins"))
print("Parameter loop:", metadata.get("parameter_loop"))
print("All metadata keys:", ", ".join(sorted(metadata)))

## Images, pools and B1 maps (NetCDF)

For Cartesian sequences the export contains `cartesian_kspace` and
`cartesian_image`; with a multi-pool phantom also one image per pool in
`species_cartesian_image`. If a B1 field was used, `tx_sensitivity_map` and
`effective_peak_b1_gauss` are given on the phantom grid.

In [ ]:
if RESULT_FILE.suffix == ".nc":
    print("signal:", dict(dataset["signal"].sizes))
    if "cartesian_image" in dataset:
        image = dataset["cartesian_image"]
        plt.imshow(np.abs(image.values), cmap="gray")
        plt.title(f"Cartesian image {dict(image.sizes)}")
        plt.colorbar()
        plt.show()
    if "species_cartesian_image" in dataset:
        for pool in range(dataset.sizes["pool"]):
            pool_image = dataset["species_cartesian_image"].isel(pool=pool)
            print(f"pool {pool}: max |image| = {float(np.abs(pool_image).max()):.4g}")
    for name in ("tx_sensitivity_map", "effective_peak_b1_gauss"):
        if name in dataset:
            print(f"{name}: {dict(dataset[name].sizes)}")

## Reconstruction methods: Cartesian FFT or non-Cartesian NUFFT

The stored images depend on the readout trajectory:

- **Cartesian** (FLASH, EPI, bSSFP, CSI): the samples lie on the k-space
  grid. The image is the centred inverse FFT of the label-sorted k-space.
- **Spiral** (non-Cartesian): the spiral layout selects the method. By
  default it is the density-compensated **NUFFT** (`"nufft"`). The
  alternatives are the iterative least-squares CG-NUFFT (`"cg"`) and linear
  gridding followed by an FFT (`"linear"`, blurrier, the legacy reference).
- **3D radial** (non-Cartesian): density-compensated trilinear gridding. The
  analysis notebook written next to a `.nc` export also shows the NUFFT of the
  same spokes.

Do not place non-Cartesian samples into an FFT as they come. A **NUFFT**
(`blochsimulator.sequence.nufft.NUFFTOperator(k_cyc_per_m, matrix, fov_m)`)
first weights the densely sampled k-space centre down (density compensation).
It then interpolates the samples onto an oversampled grid with a
Kaiser–Bessel kernel. Its `sampling_gap` is the largest gap between samples in
units of 1/FOV. Above 0.5 the trajectory is undersampled and aliases outside
`nyquist_fov_m()`.

Every format holds the raw ADC `signal` and the gradient moments
`adc_gradient_moment_cyc_per_m` `(adc, 3)` in cycles/m. A spiral export
also stores its frame layout in `metadata["spiral_acquisition"]`. With these,
`reconstruct_spiral_frames` reconstructs a spiral again with any method. It
works the same for `.nc`, `.h5` and `.npz`. This step needs the
`blochsimulator` package.

In [ ]:
if RESULT_FILE.suffix == ".nc":
    raw_signal = dataset["signal"].values  # (adc,) or (coil, adc)
    moments = (dataset["adc_gradient_moment_cyc_per_m"].transpose("adc", "gradient_axis").values
               if "adc_gradient_moment_cyc_per_m" in dataset else None)
else:
    raw_signal = arrays["signal"]
    moments = arrays.get("adc_gradient_moment_cyc_per_m")

trajectory_type = str(definitions.get("TrajectoryType", definitions.get("Trajectory", "")))
if metadata.get("spiral_acquisition") is not None and moments is not None:
    from blochsimulator.sequence.reconstruction import (
        reconstruct_spiral_frames,
        spiral_acquisition_from_metadata,
    )

    spiral = spiral_acquisition_from_metadata(metadata)
    frame = 0
    trajectory = spiral.trajectory(moments, frame)  # k = M - origin, cycles/m
    operator = spiral.nufft_operator(trajectory)
    print(f"{spiral.num_frames} spiral frame(s), stored method: {spiral.reconstruction_label}")
    print(f"Largest k-space gap: {operator.sampling_gap:.2f}/FOV (Nyquist: <= 0.5)")
    if operator.sampling_gap > 0.5:
        alias_free = " x ".join(f"{fov * 1e3:.0f}" for fov in operator.nyquist_fov_m())
        print(f"Undersampled: expect aliasing outside about {alias_free} mm.")

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].plot(trajectory[:, 0], trajectory[:, 1], lw=0.6)
    axes[0].set(title="Trajectory", xlabel="kx (1/m)", ylabel="ky (1/m)", aspect="equal")
    for ax, method in zip(axes[1:], spiral.RECONSTRUCTION_METHODS):
        # Shape (frame, [coil,] phase_y, read_x); frames=None reconstructs all.
        image = reconstruct_spiral_frames(raw_signal, moments, metadata,
                                          method=method, frames=[frame])[0]
        if image.ndim == 3:  # root sum of squares over receive coils
            image = np.sqrt(np.sum(np.abs(image) ** 2, axis=0))
        ax.imshow(np.abs(image), origin="lower", cmap="gray")
        ax.set_title(spiral.RECONSTRUCTION_LABELS[method])
        ax.axis("off")
    plt.show()
elif "radial" in trajectory_type.lower():
    print("3D radial: radial_3d_image uses trilinear gridding. For a NUFFT use "
          "NUFFTOperator(k, (N, N, N), (fov, fov, fov)) on one echo's spokes.")
else:
    print("Cartesian readout: the stored images are FFT reconstructions.")